In [1]:
import os
import glob
import numpy as np
import import_ipynb
import nibabel as nib

In [2]:
DATASET_DIR = "dataset"
NII_PATTERN  = "*.nii*"
TXT_PATTERN  = "*.txt"

In [3]:
def find_cases(dataset_dir):
    # return sorted list of (case_id, nii_path, txt_path) tuples
    cases = []
    subfolders = sorted(
        (f for f in os.scandir(dataset_dir)
         if f.is_dir() and f.name.startswith("BraTS20")),
        key=lambda x: int(x.name.split('_')[-1])
    )

    for folder in subfolders:
        nii_files = glob.glob(os.path.join(folder.path, NII_PATTERN))
        txt_files = glob.glob(os.path.join(folder.path, TXT_PATTERN))

        if not nii_files:
            print(f"[WARN] No .nii file in {folder.name}, skipping.")
            continue

        cases.append({
            "id":       folder.name,
            "nii_path": nii_files[0],   
            "txt_path": txt_files[0] if txt_files else None,
        })

    print(f"Found {len(cases)} cases.")
    return cases

In [4]:
def load_case(case):
    # load .nii volume and .txt content for one case
    img  = nib.load(case["nii_path"])
    data = img.get_fdata(dtype=np.float32)   

    # normalize to [0, 1] using robust percentile clipping
    nonzero = data[data > 0]
    if nonzero.size > 0:
        p1, p99 = np.percentile(nonzero, [1, 99])
        data = np.clip(data, p1, p99)
        data = (data - p1) / (p99 - p1 + 1e-8)

    label = None
    if case["txt_path"]:
        with open(case["txt_path"], "r") as f:
            label = f.read().strip()

    return {
        "id":     case["id"],
        "volume": data, 
        "affine": img.affine,  
        "shape":  data.shape,
        "label":  label,
    }

In [9]:
def load_dataset(dataset_dir=DATASET_DIR, max_cases=None, verbose=True):
    # load all cases into list of dicts
    cases = find_cases(dataset_dir)
    if max_cases:
        cases = cases[:max_cases]

    dataset = []
    for i, case in enumerate(cases):
        if verbose:
            print(f"Loading [{i+1}/{len(cases)}]  {case['id']} ...", end="\r")
        dataset.append(load_case(case))

    if verbose:
        print(f"\nDone. Loaded {len(dataset)} cases.")
    return dataset

In [14]:
def summarize(dataset):
    shapes = [d["shape"] for d in dataset]
    labels = [d["label"] for d in dataset]

    print(f"  total cases   : {len(dataset)}")
    print(f"  unique shapes : {set(shapes)}")
    print(f"  sample labels : {labels[:5]}")

In [15]:
def to_numpy_arrays(dataset):
    # convert to numpy array, volumes in X and labels in y, add  channel dimension so shape becomes (N, 1, H, W, D)
    X = np.stack([d["volume"] for d in dataset], axis=0)
    X = X[:, np.newaxis, ...]          # add channel dim for PyTorch/TF

    y = np.array([d["label"] for d in dataset])   # strings or None
    return X, y

In [16]:
dataset = load_dataset(max_cases=5)

Found 5 cases.
Loading [5/5]  BraTS20_Training_005 ...
Done. Loaded 5 cases.


In [17]:
summarize(dataset)

  total cases   : 5
  unique shapes : {(240, 240, 155)}
  sample labels : ['The lesion area is in the right frontal and parietal lobes with a mixed pattern of high and low signals with speckled high signal regions. Edema is mainly observed in the right parietal lobe, partially extending to the frontal lobe, presenting as high signal, indicating significant tissue swelling around the lesion. Necrosis is within the lesions of the right parietal and frontal lobes, appearing as mixed, with alternating high and low signal regions. Ventricular compression is seen in the lateral ventricles with significant compressive effects on the brain tissue and ventricles.', 'The lesion area is in the right frontal lobe, showing a mixture of high and low signal areas, with some appearing as speckled high signals, and in the right temporal lobe with prominent high-signal areas, some of which are accompanied by low signals, and in the right parietal lobe with a mix of high and low signals. Edema is mainly 

In [18]:
case = dataset[0]
print(f"case ID : {case['id']}")
print(f"volume  : {case['volume'].shape}  dtype={case['volume'].dtype}")
print(f"label   : {case['label']}")

case ID : BraTS20_Training_001
volume  : (240, 240, 155)  dtype=float64
label   : The lesion area is in the right frontal and parietal lobes with a mixed pattern of high and low signals with speckled high signal regions. Edema is mainly observed in the right parietal lobe, partially extending to the frontal lobe, presenting as high signal, indicating significant tissue swelling around the lesion. Necrosis is within the lesions of the right parietal and frontal lobes, appearing as mixed, with alternating high and low signal regions. Ventricular compression is seen in the lateral ventricles with significant compressive effects on the brain tissue and ventricles.
